# Embeddings: 
A practical notebook to test embeddings technics and validate their developement on smaller databases before tackling the 3 millions papers from arXiv.

## Dense embeddings

In [1]:
from FlagEmbedding import BGEM3FlagModel
import duckdb
import pandas as pd
import numpy as np
import gc
import os
import torch
from tqdm.notebook import tqdm
from torchao.quantization import Int8WeightOnlyConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, TorchAoConfig

/usr/local/lib/python3.12/dist-packages/torch/library.py:357: UserWarning: Warning only once for all operators,  other operators may also be overridden.
  Overriding a previously registered kernel for the same operator and the same dispatch key
  operator: flash_attn::_flash_attn_backward(Tensor dout, Tensor q, Tensor k, Tensor v, Tensor out, Tensor softmax_lse, Tensor(a6!)? dq, Tensor(a7!)? dk, Tensor(a8!)? dv, float dropout_p, float softmax_scale, bool causal, SymInt window_size_left, SymInt window_size_right, float softcap, Tensor? alibi_slopes, bool deterministic, Tensor? rng_state=None) -> Tensor
    registered at /usr/local/lib/python3.12/dist-packages/torch/_library/custom_ops.py:926
  dispatch key: ADInplaceOrView
  previous kernel: no debug info
       new kernel: registered at /usr/local/lib/python3.12/dist-packages/torch/_library/custom_ops.py:926 (Triggered internally at /opt/pytorch/pytorch/aten/src/ATen/core/dispatch/OperatorEntry.cpp:208.)
  self.m.impl(


In [2]:
# Load the BGE-M3 model 
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, device='cuda:0', max_seq_len=2048, use_cache=True)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [3]:
import duckdb
import pandas as pd

conn = duckdb.connect("data/arxiv_metadata.duckdb", read_only=True)

positives = conn.execute("""
    SELECT arxiv_id, title, abstract FROM papers
    WHERE (title ILIKE '%quantiz%' OR abstract ILIKE '%quantiz%')
      AND primary_category IN ('cs.LG', 'cs.CL', 'cs.AI', 'cs.CV')
    ORDER BY RANDOM() LIMIT 20
""").fetchdf()

noise = conn.execute("""
    SELECT arxiv_id, title, abstract FROM papers
    WHERE primary_category IN ('cs.LG', 'cs.CL', 'cs.AI', 'cs.CV')
    ORDER BY RANDOM() LIMIT 480
""").fetchdf()

conn.close()
sample = pd.concat([positives, noise], ignore_index=True).drop_duplicates("arxiv_id")
sample.head()

,arxiv_id,title,abstract
0,2306.09973,Enhancing Fault Resilience of QNNs by Selectiv...,The superior performance of Deep Neural Netw...
1,2410.17170,Self-calibration for Language Model Quantizati...,Quantization and pruning are fundamental appro...
2,2505.00980,LMDepth: Lightweight Mamba-based Monocular Dep...,Monocular depth estimation provides an addit...
3,2510.12975,A Connection Between Score Matching and Local ...,The local intrinsic dimension (LID) of data is...
4,2310.09259,QUIK: Towards End-to-End 4-Bit Inference on Ge...,Large Language Models (LLMs) from the GPT fa...


In [4]:
texts = (sample["title"] + ". " + sample["abstract"]).tolist()
doc_out = model.encode(texts, return_dense=True, return_sparse=False)
doc_dense = doc_out["dense_vecs"]   # shape: (n_docs, 1024)
doc_dense


pre tokenize: 100%|██████████| 2/2 [00:00<00:00, 62.44it/s]



Inference Embeddings: 100%|██████████| 2/2 [00:01<00:00,  1.09it/s]


array([[-0.04044 , -0.009384, -0.002243, ..., -0.01614 ,  0.0526  ,
         0.01211 ],
       [-0.03424 ,  0.001441, -0.02788 , ...,  0.006   ,  0.02768 ,
        -0.0266  ],
       [-0.0492  , -0.00777 , -0.02066 , ..., -0.04214 ,  0.00796 ,
         0.0028  ],
       ...,
       [-0.06525 , -0.02385 , -0.000766, ..., -0.0306  ,  0.02446 ,
         0.01633 ],
       [-0.06143 ,  0.02399 , -0.03577 , ...,  0.005524,  0.0504  ,
         0.03065 ],
       [-0.00714 ,  0.01334 , -0.01209 , ..., -0.03534 ,  0.03009 ,
         0.0496  ]], dtype=float16)

In [5]:
import numpy as np

query = "quantization of neural network weights for efficient inference"
query_dense = model.encode([query], return_dense=True, return_sparse=False)["dense_vecs"][0]

# Commpute the doc product scores and get the top 15 results
scores = doc_dense @ query_dense   
top_idx = np.argsort(-scores)[:15]

for i in top_idx:
    print(f"{scores[i]:.3f}  {sample.iloc[i]['title']}")

0.633  QUIK: Towards End-to-End 4-Bit Inference on Generative Large Language
  Models
0.614  Energy Efficient Hardware Acceleration of Neural Networks with
  Power-of-Two Quantisation
0.613  Joint Regularization on Activations and Weights for Efficient Neural
  Network Pruning
0.591  A2Q: Accumulator-Aware Quantization with Guaranteed Overflow Avoidance
0.588  Approximation power of random neural networks
0.564  Quamba: A Post-Training Quantization Recipe for Selective State Space
  Models
0.561  Investigating the Impact of Quantization on Adversarial Robustness
0.555  Free Hunch: Denoiser Covariance Estimation for Diffusion Models Without
  Extra Costs
0.554  Enhancing Fault Resilience of QNNs by Selective Neuron Splitting
0.554  Towards Fully 8-bit Integer Inference for the Transformer Model
0.553  Modeling Fluency and Faithfulness for Diverse Neural Machine Translation
0.543  Distilled Low Rank Neural Radiance Field with Quantization for Light
  Field Compression
0.542  Self-calibra

## Dense + sparse embedding using BGE-M3

In [6]:
conn = duckdb.connect("data/arxiv_metadata.duckdb", read_only=True)
corpus = conn.execute("""
    SELECT arxiv_id, title, abstract, submitted_date
    FROM papers
    WHERE primary_category IN ('cs.LG', 'cs.CL', 'cs.AI', 'cs.CV')
      AND submitted_date >= '2022-01-01'
    ORDER BY RANDOM()
    LIMIT 40000
""").fetchdf()
conn.close()

In [7]:

texts = (corpus["title"] + ". " + corpus["abstract"]).tolist()
batch_size = 64

all_dense, all_sparse = [], []
for i in tqdm(range(0, len(texts), batch_size)):
    out = model.encode(texts[i:i+batch_size], return_dense=True, return_sparse=True)
    all_dense.append(out["dense_vecs"])
    all_sparse.extend(out["lexical_weights"])

doc_dense = np.concatenate(all_dense, axis=0)   # shape (N, 1024) — fixed size, one row per doc
# all_sparse stays a plain list of {token_id: weight} dicts — sparse vectors are ragged by nature,
# each doc only has weights for the terms it actually contains, so they can't stack into a matrix

  0%|          | 0/625 [00:00<?, ?it/s]

In [8]:
q_out = model.encode([query], return_dense=True, return_sparse=True)
query_dense = q_out["dense_vecs"][0]
query_sparse = q_out["lexical_weights"][0]

dense_scores = doc_dense @ query_dense
sparse_scores = np.array([
    model.compute_lexical_matching_score(query_sparse, doc_sparse)
    for doc_sparse in all_sparse
])

In [9]:
def rrf_fuse(*score_arrays, k=60):
    fused = np.zeros(len(score_arrays[0]))
    for scores in score_arrays:
        ranks = np.argsort(np.argsort(-scores))   # 0 = best in that channel
        fused += 1.0 / (k + ranks + 1)
    return fused

hybrid_scores = rrf_fuse(dense_scores, sparse_scores)
top_idx = np.argsort(-hybrid_scores)[:15]

for i in top_idx:
    print(f"hybrid={hybrid_scores[i]:.5f}  dense={dense_scores[i]:.3f}  sparse={sparse_scores[i]:.3f}  {corpus.iloc[i]['title']}")

hybrid=0.03178  dense=0.660  sparse=0.232  Edge Inference with Fully Differentiable Quantized Mixed Precision
  Neural Networks
hybrid=0.03055  dense=0.631  sparse=0.233  Toward INT4 Fixed-Point Training via Exploring Quantization Error for
  Gradients
hybrid=0.02854  dense=0.643  sparse=0.209  DiscQuant: A Quantization Method for Neural Networks Inspired by
  Discrepancy Theory
hybrid=0.02845  dense=0.646  sparse=0.204  IDKM: Memory Efficient Neural Network Quantization via Implicit,
  Differentiable k-Means
hybrid=0.02817  dense=0.614  sparse=0.220  Energy Efficient Hardware Acceleration of Neural Networks with
  Power-of-Two Quantisation
hybrid=0.02807  dense=0.647  sparse=0.200  Post-training Quantization for Neural Networks with Provable Guarantees
hybrid=0.02775  dense=0.608  sparse=0.231  MARLIN: Mixed-Precision Auto-Regressive Parallel Inference on Large
  Language Models
hybrid=0.02750  dense=0.600  sparse=0.260  Precision Neural Network Quantization via Learnable Adaptive Mod

# Evaluation set

Let's build teh evaluation set to measure the precisions and recall of the retrieval.

In [10]:
GOLDEN_PATH = "data/interim/quantization_golden_set.parquet"

if os.path.exists(GOLDEN_PATH):
    to_label = pd.read_parquet(GOLDEN_PATH)
else:
    conn = duckdb.connect("data/arxiv_metadata.duckdb", read_only=True)
    conn.execute("SELECT setseed(0.42)")   # reproducible sample: reruns from scratch draw the same rows

    # Broad net for positives — independent of hybrid's own ranking, so recall isn't measured circularly
    likely_positive = conn.execute("""
        SELECT arxiv_id, title, abstract FROM papers
        WHERE (title ILIKE '%quantiz%' OR abstract ILIKE '%quantiz%')
          AND primary_category IN ('cs.LG','cs.CL','cs.AI','cs.CV')
        ORDER BY RANDOM() LIMIT 50
    """).fetchdf()

    # Hard negatives — the exact "quantification" confusion you just found in the wild
    hard_negative = conn.execute("""
        SELECT arxiv_id, title, abstract FROM papers
        WHERE (title ILIKE '%quantif%' OR abstract ILIKE '%quantif%')
          AND title NOT ILIKE '%quantiz%' AND abstract NOT ILIKE '%quantiz%'
          AND primary_category IN ('cs.LG','cs.CL','cs.AI','cs.CV')
        ORDER BY RANDOM() LIMIT 15
    """).fetchdf()

    # Plain noise floor
    random_negative = conn.execute("""
        SELECT arxiv_id, title, abstract FROM papers
        WHERE primary_category IN ('cs.LG','cs.CL','cs.AI','cs.CV')
        ORDER BY RANDOM() LIMIT 20
    """).fetchdf()

    conn.close()
    to_label = pd.concat([likely_positive, hard_negative, random_negative], ignore_index=True).drop_duplicates("arxiv_id")
    to_label["label"] = pd.NA

In [11]:
unlabeled = to_label["label"].isna()

if not unlabeled.any():
    print(f"Golden set already fully labeled ({len(to_label)} rows) — nothing to do.")
else:
    for idx, row in to_label[unlabeled].iterrows():
        if pd.notna(to_label.at[idx, "label"]):
            continue
        print(f"\n[{idx}] {row['title']}\n{row['abstract'][:550]}...")
        ans = input("Relevant? (1 = about reducing numerical precision of weights/activations, 0 = not, q = stop): ")
        if ans == "q":
            break
        to_label.at[idx, "label"] = int(ans)

    to_label.to_parquet(GOLDEN_PATH)  # merges back onto the same file, so progress is never lost across sessions

Golden set already fully labeled (85 rows) — nothing to do.


In [12]:
to_label.head()

,arxiv_id,title,abstract,label
0,1903.09940,Variational Inference with Latent Space Quanti...,Despite their tremendous success in modellin...,0
1,2106.05438,Cross-Modal Discrete Representation Learning,Recent advances in representation learning h...,0
2,2605.18856,SPHERICAL KV: Angle-Domain Attention and Rate-...,Long-context inference is increasingly constra...,1
3,2105.03536,Pareto-Optimal Quantized ResNet Is Mostly 4-bit,Quantization has become a popular technique ...,1
4,1611.06342,Quantized neural network design under weight c...,The complexity of deep neural network algori...,1


# Measuring recall and precision 

Let's measure the recall and precision of our retrieval on the manually annotated dataset. 

In [13]:
# If th ekernel stopped, you can restart from here
# Reload the model — weights are already cached locally, so this should be fast, no re-download
model = BGEM3FlagModel('BAAI/bge-m3', use_fp16=True, device='cuda:0', max_seq_len=2048, use_cache=True)

# Redefine the query and its embeddings — wiped by the restart
query = "quantization of neural network weights for efficient inference"
q_out = model.encode([query], return_dense=True, return_sparse=True)
query_dense = q_out["dense_vecs"][0]
query_sparse = q_out["lexical_weights"][0]

# Redefine the fusion helper — also wiped
def rrf_fuse(*score_arrays, k=60):
    fused = np.zeros(len(score_arrays[0]))
    for scores in score_arrays:
        ranks = np.argsort(np.argsort(-scores))
        fused += 1.0 / (k + ranks + 1)
    return fused

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

In [14]:
golden = pd.read_parquet("data/interim/quantization_golden_set_v2.parquet")
golden = golden.dropna(subset=["label"])
golden["label"] = golden["label"].astype(int)
print(golden["label"].value_counts())

label
0    65
1    20
Name: count, dtype: int64


In [15]:
g_texts = (golden["title"] + ". " + golden["abstract"]).tolist()
g_out = model.encode(g_texts, return_dense=True, return_sparse=True)
g_dense, g_sparse = g_out["dense_vecs"], g_out["lexical_weights"]

g_dense_scores = g_dense @ query_dense
g_sparse_scores = np.array([model.compute_lexical_matching_score(query_sparse, s) for s in g_sparse])
g_hybrid_scores = rrf_fuse(g_dense_scores, g_sparse_scores)

golden["dense_score"] = g_dense_scores
golden["hybrid_score"] = g_hybrid_scores

In [16]:
def evaluate_ranking(scores, labels, name):
    order = np.argsort(-scores)
    sorted_labels = np.array(labels)[order]
    n_pos = sorted_labels.sum()

    cum_hits = np.cumsum(sorted_labels)
    precision_at_k = cum_hits / (np.arange(len(sorted_labels)) + 1)
    recall_at_k = cum_hits / n_pos
    ap = precision_at_k[sorted_labels == 1].mean()   # average precision

    print(f"--- {name} ---  ({n_pos} true positives in golden set)")
    for k in [10, 20, int(n_pos)]:
        if k <= len(sorted_labels):
            print(f"  P@{k}: {precision_at_k[k-1]:.2f}   R@{k}: {recall_at_k[k-1]:.2f}")
    print(f"  Average Precision: {ap:.3f}")
    return ap

ap_dense  = evaluate_ranking(g_dense_scores,  golden["label"].values, "Dense only")
ap_hybrid = evaluate_ranking(g_hybrid_scores, golden["label"].values, "Hybrid (RRF)")

--- Dense only ---  (20 true positives in golden set)
  P@10: 0.80   R@10: 0.40
  P@20: 0.65   R@20: 0.65
  P@20: 0.65   R@20: 0.65
  Average Precision: 0.767
--- Hybrid (RRF) ---  (20 true positives in golden set)
  P@10: 0.90   R@10: 0.45
  P@20: 0.70   R@20: 0.70
  P@20: 0.70   R@20: 0.70
  Average Precision: 0.866


# Adding a LLM for classifying papers after hybrid search

Let's add a smal local LLMs to calssify each paper to increase precision. 

In [17]:
# Extract some few-shot examples to show the model, and hold out the rest for evaluation
few_shot = pd.concat([
    golden[golden["label"] == 1].sample(3, random_state=42),
    golden[golden["label"] == 0].sample(3, random_state=42),
])

# Exclude the few-shot examples from the evaluation set, so we don't score the classifier on its own training data
eval_set = golden.drop(few_shot.index)   

few_shot_block = "\n".join(
    f'Title: {r["title"]}\nAbstract: {r["abstract"][:400]}\nAnswer: {"yes" if r["label"]==1 else "no"}\n'
    for _, r in few_shot.iterrows()
)

In [18]:
del model
gc.collect()

# BGE-M3 embeddings are already computed as plain numpy — free its GPU memory before loading the LLM
torch.cuda.empty_cache()   

llm_name = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForCausalLM.from_pretrained(
    llm_name, device_map="cuda:0",
    quantization_config=TorchAoConfig(Int8WeightOnlyConfig()),
    torch_dtype="auto",
)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:1356: UserWarning: Config Deprecation: version 1 of Int8WeightOnlyConfig is deprecated and will no longer be supported in a future release, please use version 2, see https://github.com/pytorch/ao/issues/2752 for more details
  warnings.warn(


In [19]:
RUBRIC = """You are screening arXiv papers for a literature review on: quantization of neural network
weights/activations — techniques that reduce numerical precision for efficient inference
(e.g. INT8/INT4/binary/ternary weights, post-training quantization (PTQ), quantization-aware
training (QAT), mixed-precision inference).

Answer "no" when quantization (in the sense above) is not the paper's own technique or main
contribution — including these look-alikes:
- "Uncertainty quantification": measuring model confidence — an unrelated subfield.
- "Vector quantization" / discrete codebooks for representation learning (e.g. VQ-VAE-style methods):
  discretizes representations, a different technique from weight/activation precision reduction.
- Papers that merely evaluate or use already-quantized models as a side topic (e.g. studying safety
  or robustness "under quantization") without proposing or analyzing a quantization method itself.

Missing a real quantization paper is worse than including a borderline one — a human reviews the
results afterward. If genuinely uncertain after considering the paper's core contribution, answer yes.
"""

In [20]:
import re

def classify_paper(title, abstract):
    prompt = f"""{RUBRIC}
Examples:
{few_shot_block}
Now classify this paper. First, in one short sentence, name the paper's core technique. Then answer yes or no.

Title: {title}
Abstract: {abstract[:800]}

Reasoning:"""

    messages = [{"role": "user", "content": prompt}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors="pt", return_dict=True
    ).to(llm.device)
    output = llm.generate(**inputs, max_new_tokens=60, do_sample=False)
    text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    matches = re.findall(r"\b(yes|no)\b", text)   # robust to trailing punctuation/markdown around the final word
    return matches[-1] == "yes" if matches else False

In [21]:
eval_set = eval_set.copy()
eval_set["llm_predicted"] = [classify_paper(r["title"], r["abstract"]) for _, r in tqdm(eval_set.iterrows(), total=len(eval_set))]

tp = ((eval_set["llm_predicted"]) & (eval_set["label"] == 1)).sum()
fp = ((eval_set["llm_predicted"]) & (eval_set["label"] == 0)).sum()
fn = ((~eval_set["llm_predicted"]) & (eval_set["label"] == 1)).sum()
print(f"Precision: {tp/(tp+fp):.2f}   Recall: {tp/(tp+fn):.2f}   (n={len(eval_set)})")

  0%|          | 0/79 [00:00<?, ?it/s]

Precision: 0.82   Recall: 0.82   (n=79)


In [22]:
def precision_at_recall(scores, labels, target_recall):
    order = np.argsort(-scores)
    sorted_labels = np.asarray(labels)[order]
    n_pos = sorted_labels.sum()
    cum_hits = np.cumsum(sorted_labels)
    recall_at_k = cum_hits / n_pos
    precision_at_k = cum_hits / (np.arange(len(sorted_labels)) + 1)
    idx = np.searchsorted(recall_at_k, target_recall)
    if idx >= len(sorted_labels):
        return None, None
    return precision_at_k[idx], idx + 1

llm_precision, llm_recall = tp / (tp + fp), tp / (tp + fn)
hybrid_p, hybrid_k = precision_at_recall(eval_set["hybrid_score"].values, eval_set["label"].values, target_recall=llm_recall)

print(f"LLM classifier:  precision={llm_precision:.2f}  recall={llm_recall:.2f}  (n={len(eval_set)})")
print(f"Hybrid, same recall ({llm_recall:.2f}): top {hybrid_k}/{len(eval_set)} items needed, precision={hybrid_p:.2f}")

LLM classifier:  precision=0.82  recall=0.82  (n=79)
Hybrid, same recall (0.82): top 19/79 items needed, precision=0.74


# Conclusion

The LLM Qwen2.5-7B-Instruct classifier reaches **Precision 0.82 / Recall 0.82** (n=79). Compared properly, at the *same recall* (0.82) on the *same* eval set: hybrid alone needs its top 19/79 ranked items to reach that recall, at only 0.74 precision. The classifier reaches the same recall at 0.82 precision — a real ~8-point improvement. 

Thus, the hybrid search is the recall engine (cast a wide, cheap net over the full corpus); the LLM classifier is the precision cleanup on top of it (remove hybrid's false positives). This validates the two-stage retrieval architecture for production.
